# Inverter abnormal-performance screening

This notebook runs the reusable implementation in `src/anomaly_detection.py`. For each inverter, thresholds are fitted from the first 60% of chronological test rows, and flags are produced only for the later 40% during meaningful generation (`expected power >= 50`). This separates threshold calibration from the rows being screened. Flags are not confirmed hardware faults; the data has no maintenance/fault labels.


In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from anomaly_detection import score_plant

prediction_dir = PROJECT_ROOT / 'outputs' / 'predictions'
anomaly_dir = PROJECT_ROOT / 'outputs' / 'anomalies'
anomaly_dir.mkdir(parents=True, exist_ok=True)


In [2]:
plant_results = {}
for plant in (1, 2):
    predictions = pd.read_csv(prediction_dir / f'plant{plant}_test_predictions.csv')
    scores, report, summary = score_plant(predictions)
    scores.to_csv(anomaly_dir / f'plant{plant}_anomaly_scores.csv', index=False)
    report.to_csv(anomaly_dir / f'plant{plant}_anomaly_report.csv', index=False)
    summary.to_csv(anomaly_dir / f'plant{plant}_inverter_anomaly_summary.csv', index=False)
    plant_results[plant] = (scores, report, summary)
    print(f'Plant {plant}: {len(report):,} flags / {len(scores):,} evaluated active rows')
    display(summary)


Plant 1: 113 flags / 2,022 evaluated active rows


,SOURCE_KEY,Total_Observations,Anomalies,Mean_Residual,Mean_Deviation_Pct,P95_Abs_Norm_Residual,Calibration_Observations,Normal,Anomaly_Percentage,Peer_Deviation_Pct,ABS_Peer_Deviation_Pct
0,1IF53ai7Xc0U56Y,92,7,-1.495322,3.476218,0.594035,140,85,7.608696,0.026624,0.026624
1,adLQvlD726eNBSB,92,7,-1.510986,3.638918,0.596723,140,85,7.608696,0.189324,0.189324
2,zBIq5rxdHJRwDNY,90,6,-5.396925,2.659039,0.631059,140,84,6.666667,-0.790555,0.790555
3,iCRJl6heRkivqQ3,92,6,-1.469973,3.426829,0.631354,140,86,6.521739,-0.022764,0.022764
4,7JYdWkrLSPkdwr4,92,6,-1.465217,3.795134,0.590575,140,86,6.521739,0.345540,0.345540
5,3PZuoBAID5Wc2HD,92,6,-1.477601,3.689291,0.648387,140,86,6.521739,0.239697,0.239697
6,uHbuxQJl8lW7ozc,92,6,-1.487616,3.271528,0.633072,140,86,6.521739,-0.178066,0.178066
7,zVJPv84UY57bAof,92,6,-1.494002,3.221678,0.623954,140,86,6.521739,-0.227915,0.227915
8,pkci93gMrogZuBj,92,6,-1.469526,3.356202,0.586397,140,86,6.521739,-0.093392,0.093392
9,WRmjgnKYAwPKWDb,92,6,-1.461898,3.656370,0.639541,140,86,6.521739,0.206776,0.206776


Plant 2: 78 flags / 1,998 evaluated active rows


,SOURCE_KEY,Total_Observations,Anomalies,Mean_Residual,Mean_Deviation_Pct,P95_Abs_Norm_Residual,Calibration_Observations,Normal,Anomaly_Percentage,Peer_Deviation_Pct,ABS_Peer_Deviation_Pct
0,81aHJ1q11NBPMrL,91,7,-7.382271,3.986144,0.798873,143,84,7.692308,-1.856699,1.856699
1,9kRcWv60rDACzjR,92,7,-6.660652,8.288016,0.837612,127,85,7.608696,2.445173,2.445173
2,WcxssY2VbP4hApt,92,7,-6.856812,4.811278,0.813128,142,85,7.608696,-1.031565,1.031565
3,q49J1IKaHRwDQnt,91,6,-2.083291,9.454501,0.861770,143,85,6.593407,3.611658,3.611658
4,vOuJvMaM2sgwLmb,91,6,-7.464396,3.763026,0.789622,142,85,6.593407,-2.079817,2.079817
5,LYwnQax7tkwH5Cb,92,6,-6.311304,4.837053,0.854048,143,86,6.521739,-1.005790,1.005790
6,LlT2YUhhzqhg5Sw,92,6,-6.530005,4.806671,0.884917,127,86,6.521739,-1.036172,1.036172
7,xoJJ8DcxJEcupym,90,5,-8.441762,5.790890,0.819570,143,85,5.555556,-0.051954,0.051954
8,4UPUqMRk7TRMgml,90,4,-11.312667,7.197133,1.000000,105,86,4.444444,1.354290,1.354290
9,Qf4GUc1pJu5T6c6,90,3,-11.220815,7.627276,1.000000,91,87,3.333333,1.784432,1.784432


In [3]:
# Inspect flagged observations from either plant.
display(plant_results[1][1].head(20))
display(plant_results[2][1].head(20))


,DATE_TIME,SOURCE_KEY,ACTUAL_AC_POWER,EXPECTED_AC_POWER,RESIDUAL,NORMALIZED_RESIDUAL,ABS_NORMALIZED_RESIDUAL,P95_Abs_Norm_Residual,CALIBRATION_OBSERVATIONS
0,2020-06-16 14:15:00,VHMLBKoKgIrUVDU,1032.025000,423.442857,608.582143,1.437224,1.437224,0.612952,140
1,2020-06-16 14:15:00,ZoEaEvLYb1n2sOq,1051.828571,432.925000,618.903571,1.429586,1.429586,0.631224,140
2,2020-06-16 14:15:00,7JYdWkrLSPkdwr4,1000.412500,414.571429,585.841071,1.413125,1.413125,0.590575,140
3,2020-06-16 14:15:00,adLQvlD726eNBSB,1050.128571,439.037500,611.091071,1.391888,1.391888,0.596723,140
4,2020-06-16 14:15:00,YxYtjZvoooNbGkE,1036.642857,435.425000,601.217857,1.380761,1.380761,0.621455,140
5,2020-06-17 15:45:00,rGa61gmuvPhdLxV,329.075000,140.471429,188.603571,1.342647,1.342647,0.626502,140
6,2020-06-17 15:45:00,ZoEaEvLYb1n2sOq,317.985714,136.450000,181.535714,1.330419,1.330419,0.631224,140
7,2020-06-17 15:45:00,3PZuoBAID5Wc2HD,328.350000,140.957143,187.392857,1.329431,1.329431,0.648387,140
8,2020-06-17 15:45:00,zBIq5rxdHJRwDNY,317.050000,136.185714,180.864286,1.328071,1.328071,0.631059,140
9,2020-06-17 15:45:00,YxYtjZvoooNbGkE,315.542857,136.212500,179.330357,1.316548,1.316548,0.621455,140


,DATE_TIME,SOURCE_KEY,ACTUAL_AC_POWER,EXPECTED_AC_POWER,RESIDUAL,NORMALIZED_RESIDUAL,ABS_NORMALIZED_RESIDUAL,P95_Abs_Norm_Residual,CALIBRATION_OBSERVATIONS
0,2020-06-17 13:45:00,xoJJ8DcxJEcupym,373.833333,56.833333,317.000000,5.577713,5.577713,0.819570,143
1,2020-06-17 13:45:00,oZZkBaNadn6DNKz,371.200000,56.493333,314.706667,5.570687,5.570687,1.115118,114
2,2020-06-17 13:45:00,PeE6FRyGXUgsRhN,362.666667,55.546667,307.120000,5.529045,5.529045,1.005090,130
3,2020-06-17 13:45:00,9kRcWv60rDACzjR,366.133333,57.220000,308.913333,5.398695,5.398695,0.837612,127
4,2020-06-16 14:45:00,q49J1IKaHRwDQnt,512.760000,83.942857,428.817143,5.108441,5.108441,0.861770,143
5,2020-06-17 13:45:00,oZ35aAeoifZaQzV,351.550000,57.760000,293.790000,5.086392,5.086392,1.000000,97
6,2020-06-17 13:45:00,Mx2yZCDsyf6DPfv,348.464286,57.340000,291.124286,5.077159,5.077159,1.000000,96
7,2020-06-17 13:45:00,Qf4GUc1pJu5T6c6,344.971429,56.800000,288.171429,5.073441,5.073441,1.000000,91
8,2020-06-17 13:45:00,V94E5Ben1TlhnDV,319.100000,56.020000,263.080000,4.696180,4.696180,1.084264,98
9,2020-06-17 13:45:00,mqwcsP2rE7J0TFp,317.393333,59.700000,257.693333,4.316471,4.316471,1.018851,137
